This is an analysis of the *San Francisco Salaries* dataset acquired from the following link at [Kaggle](https://www.kaggle.com/datasets/kaggle/sf-salaries?resource=download).

THe data is for San Francisco city employees from 2011-2014. This allows for a comparisons in one broad category: how compensation is distributed and how it changed over the four-year period. This encompases a variety of aspects.

Compensation distribution can refer to the whole government budger and within specific groups or positions. For example, looking at what portion of a position's compensation is overtime, and what portion of the whole budget goes towards IT Workers in general.

In [1]:
import sqlite3 as sql
import pandas as pd

In [2]:
con = sql.connect("Salaries.sqlite")
cur = con.cursor()

In [3]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
cur.execute(query)
tables = cur.fetchall()

In [4]:
columnDict = {}

for i,table in enumerate(tables):
    query = "SELECT * FROM %s;" % table
    cur.execute(query)
    cols = list(cur.description)
    valuelist = []
    for j, col in enumerate(cols):
        collist = list(col)
        valuelist.append(collist[0])
    columnDict[table] = valuelist

columnDict

{('Salaries',): ['Id',
  'EmployeeName',
  'JobTitle',
  'BasePay',
  'OvertimePay',
  'OtherPay',
  'Benefits',
  'TotalPay',
  'TotalPayBenefits',
  'Year',
  'Notes',
  'Agency',
  'Status']}

In [5]:
query = """ SELECT DISTINCT Year from Salaries"""
cur.execute(query)
cur.fetchall()

[(2011,), (2012,), (2013,), (2014,)]

In [6]:
query = """ SELECT Year, COUNT(Year) from Salaries GROUP By Year"""
cur.execute(query)
cur.fetchall()

[(2011, 36159), (2012, 36766), (2013, 37606), (2014, 38123)]

In [7]:
query = """ SELECT DISTINCT Notes from Salaries"""
cur.execute(query)
cur.fetchall()

[('',)]

In [8]:
query = """ SELECT DISTINCT Agency from Salaries"""
cur.execute(query)
cur.fetchall()

[('San Francisco',)]

In [9]:
query = """ SELECT DISTINCT Status from Salaries"""
cur.execute(query)
cur.fetchall()

[('',), ('PT',), ('FT',)]

In [10]:
query = """ SELECT Status, COUNT(Status) from Salaries GROUP By Status"""
cur.execute(query)
cur.fetchall()

[('', 110535), ('FT', 22334), ('PT', 15785)]

In [11]:
query = """SELECT DISTINCT JobTitle from Salaries"""
cur.execute(query)
cur.fetchmany(5)

[('GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY',),
 ('CAPTAIN III (POLICE DEPARTMENT)',),
 ('WIRE ROPE CABLE MAINTENANCE MECHANIC',),
 ('DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)',),
 ('ASSISTANT DEPUTY CHIEF II',)]

In [12]:
query = """SELECT COUNT(DISTINCT JobTitle) from Salaries"""
cur.execute(query)
cur.fetchmany(5)

[(2159,)]

In [13]:
query = """ SELECT DISTINCT Status from Salaries"""
cur.execute(query)
cur.fetchall()

[('',), ('PT',), ('FT',)]

In [14]:
query = """ SELECT MIN(BasePay), MAX(BasePay), AVG(BasePay) from Salaries"""
cur.execute(query)
cur.fetchall()

[(-166.01, 'Not Provided', 66053.72928807836)]

In [15]:
query="Select JobTitle,BasePay from Salaries where BasePay < 0"
cur.execute(query)
cur.fetchall()

[('Junior Clerk', -166.01),
 ('Junior Clerk', -121.63),
 ('Junior Clerk', -109.22),
 ('Junior Clerk', -106.6),
 ('Junior Clerk', -101.88),
 ('Junior Clerk', -93.14),
 ('Junior Clerk', -87.38),
 ('Junior Clerk', -75.67),
 ('Junior Clerk', -59.59),
 ('Junior Clerk', -30.58),
 ('Clerk', -9.5)]

In [16]:
query="Select Count(JobTitle) from Salaries where JobTitle in ('Junior Clerk')"
cur.execute(query)
cur.fetchall()

[(596,)]

In [17]:
query = """ SELECT MIN(OvertimePay), MAX(OvertimePay), AVG(OvertimePay) from Salaries where OvertimePay Not in ('Not Provided')"""
cur.execute(query)
cur.fetchall()

[(-0.01, 245131.88, 5066.059886444668)]

In [18]:
query="Select JobTitle,OvertimePay from Salaries where OvertimePay < 0"
cur.execute(query)
cur.fetchall()

[('Senior Eligibility Worker', -0.01)]

In [19]:
query = """ SELECT MIN(OtherPay), MAX(OtherPay), AVG(OtherPay) from Salaries where OtherPay Not in ('Not Provided')"""
cur.execute(query)
cur.fetchall()

[(-7058.59, 400184.25, 3648.767296804574)]

In [20]:
query="Select JobTitle,OtherPay from Salaries where OtherPay < 0"
cur.execute(query)
cur.fetchall()

[('IS Business Analyst-Principal', -7058.59),
 ('Custodial Supervisor', -9.6),
 ('Gardener', -46.76),
 ('Special Nurse', -50.19),
 ('Counselor, Log Cabin Ranch', -618.13)]

In [21]:
query = """ SELECT MIN(Benefits), MAX(Benefits), AVG(Benefits) from Salaries where Benefits Not in ('Not Provided')"""
cur.execute(query)
cur.fetchall()

[(-33.89, '', 18924.742068146654)]

In [22]:
query="Select JobTitle,Benefits from Salaries where Benefits < 0"
cur.execute(query)
cur.fetchall()

[('Police Officer 3', -2.73),
 ('Police Officer 3', -8.2),
 ('Police Officer 3', -33.89),
 ('Secretary 2', -13.8)]

In [23]:
query = """ SELECT MIN(TotalPay), MAX(TotalPay), AVG(TotalPay) from Salaries"""
cur.execute(query)
cur.fetchall()

[(-618.13, 567595.43, 74768.32197169265)]

In [24]:
query = """ SELECT MIN(TotalPayBenefits), MAX(TotalPayBenefits), AVG(TotalPayBenefits) from Salaries"""
cur.execute(query)
cur.fetchall()

[(-618.13, 567595.43, 93692.55481056681)]

# Data Exploration

There are some strange outliers such as jobs with negative base pay that will likely pop up as the exploration and analysis continues.

With the broad strokes established, mainly what possible values there are for each category, the focus can be shifted on comparing and contrasting different groups of employees. Most of these are curiousities outside of a handful.

The benefits will be looked at over time, as well as the distribution of job titles and if they change in name. Some positions may be renamed and won't be picked up by the **LIKE** command paramaters set up. 

I'll be looking at the following:

* **Law Enforcement**: by looking at the strings *Police* and *Sheriff* I can see the Law Enforcement wages paid by the city.
* **Fire Department**: By taking a look at the string *Fire* I can see how much is spent on the Fire Department in terms of wages.
* **Mayor**: by querying 'Mayor' to take a look at the mayor and their staff.
* **Healthcare**: This one is a variety of strings that are cut off to bring in the largest variety of job titles. The list is *nurs* (gets nurse and nursery), *medical*, *health*, *dent*, *pharma* (pharmacy and pharmacist), *Physician*, and *Social Work*. This should get most healthcare workers though with so many strings it will probably get a few incorrect ones which is why I will also try to avoid the strings *resources* and *examiner*. 
* **Information Science**: All of the IT/IS positions start with 'IS' so looking for that with a space afterwards should get all of those positions without having to filter too many other job titles.
* **Justice**: *Defender* should pick up Public Defenders and *court* should pick up most who work at the court. Judge did not seem to get many results which is strange, and even looking for it directly did not give any results aside from a secretary.
* **Engineers and Scientists**: Looking at the strings *bio*, *chem*, *research* and *engineer* should get all the relevant job titles. This is to see how much money is spent on research, development, and engineers in general.
* **Planning**: Looking at the string *plann* should get all job titles related to urban planning.
* **Technicians and Mechanics**: *automotive*, *Electr*, *mech*, *plumb* should get all handymen, mechanics, and technicians. This is to see how much is spent on maintenance work and similar tasks.
* **Clerks**: *clerk* and *secretar* should bring up all clerks, secretaries and other clerical staff to see how much is spent on office staff.

In [25]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%Judge%'"""
cur.execute(query)
cur.fetchall()

[('SECRETARY TO THE PRESIDING JUDGE', 2904666.62, 2011)]

## Law Enforcement Jobs

In [ ]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle), Year from Salaries 
WHERE JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%' 
Group by JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('AIRPORT POLICE SERVICES AIDE', 11244297.15, 60129.93128342246, 187, 2011),
 ('ASSISTANT INSPECTOR (POLICE DEPARTMENT)', 188999.2, 188999.2, 1, 2011),
 ('ASSISTANT INSPECTOR II (POLICE DEPARTMENT)',
  1508888.61,
  150888.861,
  10,
  2011),
 ('ASSISTANT INSPECTOR III (POLICE DEPARTMENT)',
  1374040.94,
  152671.21555555554,
  9,
  2011),
 ('ASSISTANT SHERIFF', 189675.09, 94837.545, 2, 2011),
 ('Assistant Sheriff', 767179.29, 191794.8225, 4, 2012),
 ('CAPTAIN III (POLICE DEPARTMENT)', 7836004.05, 211783.89324324325, 37, 2011),
 ('CHIEF DEPUTY SHERIFF', 503875.24, 167958.41333333333, 3, 2011),
 ('CHIEF OF POLICE', 267992.59, 267992.59, 1, 2011),
 ('COMMANDER III, (POLICE DEPARTMENT)',
  1435955.55,
  205136.50714285715,
  7,
  2011),
 ('Chief Deputy Sheriff', 2181134.48, 218113.448, 10, 2012),
 ('Chief of Police', 1235196.8, 411732.26666666666, 3, 2012),
 ('Community Police Services Aide', 51794326.06, 92655.32389982112, 559, 2012),
 ('DEPUTY CHIEF III (POLICE DEPARTMENT)',
  1250132.

In [ ]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Year = '2011' AND (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('AIRPORT POLICE SERVICES AIDE', 11244297.15, 60129.93128342246, 187),
 ('ASSISTANT INSPECTOR (POLICE DEPARTMENT)', 188999.2, 188999.2, 1),
 ('ASSISTANT INSPECTOR II (POLICE DEPARTMENT)', 1508888.61, 150888.861, 10),
 ('ASSISTANT INSPECTOR III (POLICE DEPARTMENT)',
  1374040.94,
  152671.21555555554,
  9),
 ('ASSISTANT SHERIFF', 189675.09, 94837.545, 2),
 ('CAPTAIN III (POLICE DEPARTMENT)', 7836004.05, 211783.89324324325, 37),
 ('CHIEF DEPUTY SHERIFF', 503875.24, 167958.41333333333, 3),
 ('CHIEF OF POLICE', 267992.59, 267992.59, 1),
 ('COMMANDER III, (POLICE DEPARTMENT)', 1435955.55, 205136.50714285715, 7),
 ('DEPUTY CHIEF III (POLICE DEPARTMENT)', 1250132.44, 250026.48799999998, 5),
 ('DEPUTY SHERIFF', 64951787.99, 94820.12845255475, 685),
 ('INSPECTOR II, (POLICE DEPARTMENT)',
  432848.20999999996,
  144282.73666666666,
  3),
 ('INSPECTOR III, (POLICE DEPARTMENT)', 24968457.92, 154126.28345679014, 162),
 ('INSPECTOR, (POLICE DEPARTMENT)', 318997.97, 159498.985, 2),
 ('INSTITUTIONAL 

In [ ]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Year = '2012' AND (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('Assistant Sheriff', 261249.93, 130624.965, 2),
 ('Chief Deputy Sheriff', 772327.6, 193081.9, 4),
 ('Chief of Police', 391362.3, 391362.3, 1),
 ('Community Police Services Aide', 17022524.97, 95097.90486033518, 179),
 ('Deputy Sheriff', 90260344.97, 136139.2835143288, 663),
 ('Deputy Sheriff 1', 291234.74, 32359.415555555555, 9),
 ('Inspector, (Police Department)', 389651.99, 194825.995, 2),
 ('Institutional Police Officer', 1284354.38, 107029.53166666666, 12),
 ('Institutional Police Sergeant', 349038.82999999996, 174519.41499999998, 2),
 ('Police Officer', 51445336.71, 122781.23319809069, 419),
 ('Police Officer 2', 68668793.7, 162337.57375886524, 423),
 ('Police Officer 3', 138076018.47, 159810.2065625, 864),
 ('Police Services Aide', 60968.55, 60968.55, 1),
 ('Senior Deputy Sheriff', 13674136.34, 155387.91295454546, 88),
 ('Sergeant, (Police Department)', 2842573.24, 177660.8275, 16),
 ('Sheriff', 19891.46, 19891.46, 1),
 ('Sheriff (SFERS)', 256636.3, 256636.3, 1),
 ("Sheriff's C

In [ ]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Year = '2013' AND (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('Assistant Sheriff', 246135.72, 246135.72, 1),
 ('Chief Deputy Sheriff', 702560.07, 234186.68999999997, 3),
 ('Chief of Police', 425815.28, 425815.28, 1),
 ('Community Police Services Aide', 16505787.14, 93253.03468926554, 177),
 ('Deputy Sheriff', 93750724.8, 143569.25696784072, 653),
 ('Deputy Sheriff 1', 2388737.16, 54289.48090909091, 44),
 ('Inspector, (Police Department)', 428154.08999999997, 142718.03, 3),
 ('Institutional Police Officer', 1324675.09, 101898.08384615385, 13),
 ('Institutional Police Sergeant', 320286.06000000006, 160143.03000000003, 2),
 ('Police Officer', 64347754.07, 127927.94049701789, 503),
 ('Police Officer 2', 68171160.25, 177992.5855091384, 383),
 ('Police Officer 3', 134598825.71, 173900.2916149871, 774),
 ('Senior Deputy Sheriff', 13398366.82, 161426.10626506025, 83),
 ('Sergeant, (Police Department)', 2605813.9, 200447.22307692308, 13),
 ('Sheriff (SFERS)', 298551.11, 298551.11, 1),
 ("Sheriff's Cadet", 2725059.54, 56772.07375, 48),
 ("Sheriff's Capta

In [ ]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Year = '2014' AND (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('Assistant Sheriff', 259793.64, 259793.64, 1),
 ('Chief Deputy Sheriff', 706246.81, 235415.60333333336, 3),
 ('Chief of Police', 418019.22, 418019.22, 1),
 ('Community Police Services Aide', 18266013.95, 89980.36428571428, 203),
 ('Deputy Sheriff', 92239944.64, 149497.4791572123, 617),
 ('Deputy Sheriff (SFERS)', 1981418.08, 123838.63, 16),
 ('Deputy Sheriff 1', 2335930.6, 93437.224, 25),
 ('Inspector, (Police Department)', 220410.34, 220410.34, 1),
 ('Institutional Police Officer', 1263729.57, 105310.7975, 12),
 ('Institutional Police Sergeant', 361014.1, 180507.05, 2),
 ('Lieutenant (Police Department)', 186913.98, 186913.98, 1),
 ('Police Officer', 67365805.01, 121598.92601083033, 554),
 ('Police Officer 2', 58739406.98, 175341.5133731343, 335),
 ('Police Officer 3', 133415475.56, 170390.1348148148, 783),
 ('Senior Deputy Sheriff', 13298249.2, 168332.26835443036, 79),
 ('Sergeant, (Police Department)', 1774740.37, 197193.37444444446, 9),
 ('Sheriff (SFERS)', 300529.16, 300529.16, 

In [56]:
query = """SELECT SUM(TotalPayBenefits), AVG(TotalPayBenefits), Count(Year), Year from Salaries 
WHERE JobTitle LIKE '%Police%' OR  JobTitle LIKE '%Sheriff%' 
Group by Year Order by Year"""
cur.execute(query)
cur.fetchall()

[(410140218.78, 118127.9431970046, 3472, 2011),
 (405868151.22, 142359.92676955456, 2851, 2012),
 (420047857.72, 149589.69292022794, 2808, 2013),
 (413901671.01, 146721.61326125488, 2821, 2014)]

### First Discussion

Interestingly, 2011 seems to be anomalous compared to the other three years. 2012-2014 all have and average pay and benefits between 140 and 150 thousand, with a total staff of around 2800, and a total budget for staff of 400-420 million. 2011 has a similar budget with 600 more staff and an average salary with benefits of a bit below 120000. Quickly looking over the job titles for each year reveals that 2011 uniquely has *AIRPORT POLICE SERVICES AIDE*, *'POLICE COMMUNICATIONS SHIFT SUPERVISOR*, and *SENIOR POLICE COMMUNICATIONS DISPATCHER* which account for around 200 employees and around 15 million in pay and benefits. An additional 45 million in pay and benefits can be found with the Sergeant III position found in the 2011 data set across 300 employees. The first set of positions have a lower average pay and benefits than 140,000. 

It's highly likely that the positions were renamed so trying to match **Police** and **Sheriff** won't pick it up, though the Sergeant III position was probably just absorbed under another job title. This means it would be good to rerun the previous SQL query removing the aforementioned positions to see how the numbers compare. 

In [32]:
query = """SELECT  SUM(TotalPayBenefits), AVG(TotalPayBenefits), Count(Year), Year from Salaries 
WHERE (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
AND (JobTitle NOT LIKE '%communications%' AND Jobtitle NOT LIKE '%airport%') 
GROUP By Year Order by Year"""
cur.execute(query)
cur.fetchall()

[(395799064.1, 121672.01478635106, 3253, 2011),
 (405868151.22, 142359.92676955456, 2851, 2012),
 (420047857.72, 149589.69292022794, 2808, 2013),
 (413901671.01, 146721.61326125488, 2821, 2014)]

In [ ]:
query = """SELECT JobTitle, COUNT(JobTitle) from Salaries 
WHERE Jobtitle LIKE '%Captain%' OR Jobtitle LIKE '%Lieutenant%' OR Jobtitle LIKE '%cadet%' 
GROUP By JobTitle Order by JobTitle"""
df = pd.read_sql_query(query, con)
df.head()

,JobTitle,COUNT(JobTitle)
0,CAPTAIN III (POLICE DEPARTMENT),37
1,"CAPTAIN, BUREAU OF FIRE PREVENTION AND PUBLIC ...",1
2,"CAPTAIN, EMERGENCYCY MEDICAL SERVICES",24
3,"CAPTAIN, FIRE SUPPRESSION",71
4,Captain 3,101


In [ ]:
query = """SELECT JobTitle, COUNT(JobTitle) from Salaries 
WHERE Jobtitle LIKE '%Captain%' OR Jobtitle LIKE '%CPT%' OR Jobtitle LIKE '%cadet%' 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('CAPTAIN III (POLICE DEPARTMENT)', 37),
 ('CAPTAIN, BUREAU OF FIRE PREVENTION AND PUBLIC SAFE', 1),
 ('CAPTAIN, EMERGENCYCY MEDICAL SERVICES', 24),
 ('CAPTAIN, FIRE SUPPRESSION', 71),
 ('Captain 3', 101),
 ('Captain, (Fire Department)', 1),
 ('Captain, Emergency Med Svcs', 75),
 ('Captain, Fire Suppression', 211),
 ("SHERIFF'S CADET", 64),
 ("SHERIFF'S CAPTAIN", 8),
 ("Sheriff's Cadet", 186),
 ("Sheriff's Captain", 23)]

In [ ]:
query = """SELECT JobTitle, COUNT(JobTitle) from Salaries 
WHERE Jobtitle LIKE '%Sergeant%' 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('INSTITUTIONAL POLICE SERGEANT', 2),
 ('Institutional Police Sergeant', 6),
 ('SERGEANT I (POLICE DEPARTMENT)', 17),
 ('SERGEANT II (POLICE DEPARTMENT)', 19),
 ('SERGEANT III (POLICE DEPARTMENT)', 295),
 ("SHERIFF'S SERGEANT", 55),
 ('Sergeant 2', 131),
 ('Sergeant 3', 1047),
 ('Sergeant, (Police Department)', 38),
 ("Sheriff's Sergeant", 153)]

In [ ]:
query = """SELECT JobTitle, COUNT(JobTitle) from Salaries 
WHERE Year='2012' AND (Jobtitle LIKE '%Sergeant%' OR Jobtitle LIKE '%SGT%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('Institutional Police Sergeant', 2),
 ('Sergeant 2', 22),
 ('Sergeant 3', 317),
 ('Sergeant, (Police Department)', 16),
 ("Sheriff's Sergeant", 50)]

In [ ]:
query = """SELECT JobTitle, COUNT(JobTitle) from Salaries 
WHERE Year='2013' AND (Jobtitle LIKE '%Sergeant%' OR Jobtitle LIKE '%SGT%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('Capt,Fire Prev or Fire Invsgtn', 7),
 ('Institutional Police Sergeant', 2),
 ('Sergeant 2', 55),
 ('Sergeant 3', 361),
 ('Sergeant, (Police Department)', 13),
 ("Sheriff's Sergeant", 53)]

In [ ]:
query = """SELECT JobTitle, COUNT(JobTitle) from Salaries 
WHERE Year='2014' AND (Jobtitle LIKE '%Sergeant%' OR Jobtitle LIKE '%SGT%') 
GROUP By JobTitle Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[('Capt,Fire Prev or Fire Invsgtn', 7),
 ('Institutional Police Sergeant', 2),
 ('Sergeant 2', 54),
 ('Sergeant 3', 369),
 ('Sergeant, (Police Department)', 9),
 ("Sheriff's Sergeant", 50)]

In [ ]:
query = """SELECT Year, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Jobtitle LIKE '%Sergeant%' OR Jobtitle LIKE '&SGT%' 
GROUP By Year Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[(2011, 56731648.09, 146215.58786082474, 388),
 (2012, 76630901.02, 188282.31208845208, 407),
 (2013, 98967307.4, 204477.9078512397, 484),
 (2014, 97891786.55, 202255.75733471074, 484)]

In [ ]:
query = """SELECT Year, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Jobtitle LIKE '%lieutenant%' 
GROUP By Year Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[(2011, 51151974.97, 161873.33851265823, 316),
 (2012, 70384376.46, 213285.98927272725, 330),
 (2013, 74491359.92, 226417.5073556231, 329),
 (2014, 71893239.42, 219857.00128440367, 327)]

In [ ]:
query = """SELECT Year, SUM(TotalPayBenefits), AVG(TotalPayBenefits), COUNT(JobTitle) from Salaries 
WHERE Jobtitle LIKE '%captain%' 
GROUP By Year Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[(2011, 26186044.27, 185716.6260283688, 141),
 (2012, 33575285.82, 243299.17260869566, 138),
 (2014, 34926113.26, 247702.9309219858, 141),
 (2013, 33892915.55, 256764.51174242422, 132)]

The difference in employment from year to year becomes less significant when taking a title like *sergeant* or *captian* makes the difference smaller. There is a somewhat constant amount of captains, lieutenants, and sergeants over time though the benefits are much worse in 2011. This may be due to budget cuts in 2011 and 2012 as per a [San Francisco Gate Article](https://www.sfgate.com/bayarea/article/Police-staffing-expected-to-shrink-2459144.php#ixzz1Gb2OcXrN), or that there is any aspect that I am missing for this. I figure budget cuts are more likely since the average pay benefits are much lower even for effectively the same position (within sergeants or captains for example).

Aside from the quirks of 2011, the distribution and compenssation of posiitions is basically consistent. There are differences, of course, like the fact that 2014 had an uptick in the number of Community Police Services Aides, and Sheriff's Cadets (each increased by about 30) as well as a loss of around 50 Police Officer II's without that being fully reflected in other roles such as Police Officer III's (which did have a bit of an uptick), or Police Sergeants. The changes between 2012 and 2013 are fairly minimal, with a decently signficant decrease of Sheriff's Cadets (from 60 down to 48), albeit with a parallel increase of Deputy Sherriff 1's from 9 to 44.

Basically, the number of personnel is basically constant, which can be seen from the previous look into captains, sergeants, and lieutenants, and from ignoring the positions such as *AIRPORT POLICE SERVICES AIDE* in 2011 which were likely absorbed into other positions either in or out of the police department.

What's left is taking a look at Status (Number of full time versus part time employees), portion of overtime pay, benefits pay, and base pay. It is important to note that most jobs are not actually labeled as full or part time so that will be a minimally important aspect of the comparison.

In [ ]:
query = """SELECT AVG(TotalPayBenefits), Status, COUNT(Status) from Salaries 
WHERE (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
GROUP By Status Order by JobTitle"""
cur.execute(query)
cur.fetchall()

[(135369.2068470047, '', 9131),
 (163835.18180519482, 'FT', 2310),
 (69358.90614481409, 'PT', 511)]

In [55]:
query = """SELECT Year, JobTitle, AVG(BasePay), AVG(OvertimePay), AVG(OtherPay), AVG(Benefits), COUNT(JobTitle) from Salaries 
WHERE (JobTitle LIKE '%Police%' OR JobTitle LIKE '%Sheriff%') 
GROUP By JobTitle Order by JobTitle"""
df = pd.read_sql_query(query, con)
df.head()

,Year,JobTitle,AVG(BasePay),AVG(OvertimePay),AVG(OtherPay),AVG(Benefits),COUNT(JobTitle)
0,2011,AIRPORT POLICE SERVICES AIDE,51178.742834,5478.936096,3472.252353,0.0,187
1,2011,ASSISTANT INSPECTOR (POLICE DEPARTMENT),123169.950000,45679.840000,20149.410000,0.0,1
2,2011,ASSISTANT INSPECTOR II (POLICE DEPARTMENT),127866.102000,13547.629000,9475.130000,0.0,10
3,2011,ASSISTANT INSPECTOR III (POLICE DEPARTMENT),130479.954444,13373.892222,8817.368889,0.0,9
4,2011,ASSISTANT SHERIFF,94400.405000,0.000000,437.140000,0.0,2


## Fire Department Jobs

In [44]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%Fire%' Group by JobTitle Order by Year"""
cur.execute(query)
cur.fetchall()

[('ASSISTANT CHIEF OF DEPARTMENT, (FIRE DEPARTMENT)', 610283.55, 2011),
 ('BATTALION CHIEF, (FIRE DEPARTMENT)', 9749499.16, 2011),
 ('CAPTAIN, BUREAU OF FIRE PREVENTION AND PUBLIC SAFE', 206704.63, 2011),
 ('CAPTAIN, FIRE SUPPRESSION', 12762877.81, 2011),
 ('CHIEF FIRE ALARM DISPATCHER', 112798.37, 2011),
 ('CHIEF OF DEPARTMENT, (FIRE DEPARTMENT)', 302377.73, 2011),
 ('DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)', 838078.68, 2011),
 ('FIRE ALARM DISPATCHER', 130371.56999999999, 2011),
 ('FIRE FIGHTER PARAMEDIC', 38957981.84, 2011),
 ('FIRE PROTECTION ENGINEER', 507207.46, 2011),
 ('FIRE RESCUE PARAMEDIC', 566616.29, 2011),
 ('FIRE SAFETY INSPECTOR II', 1678561.17, 2011),
 ('FIREFIGHTER', 110597884.48, 2011),
 ('INSPECTOR, BUREAU OF FIRE PREVENTION AND PUBLIC SA', 3220814.25, 2011),
 ('INVESTIGATOR, BUREAU OF FIRE INVESTIGATION', 702051.63, 2011),
 ('LIEUTENANT, BUREAU OF FIRE PREVENTION AND PUBLIC S', 1534524.46, 2011),
 ('LIEUTENANT, FIRE DEPARTMENT', 27905707.74, 2011),
 ('MARINE EN

In [45]:
query = """SELECT SUM(TotalPayBenefits), AVG(TotalPayBenefits), Count(Year), Year from Salaries WHERE JobTitle LIKE '%Fire%' Group by Year Order by Year"""
cur.execute(query)
cur.fetchall()

[(211563265.94, 145005.66548320767, 1459, 2011),
 (273810371.21, 188965.05949620425, 1449, 2012),
 (291038363.49, 199341.3448561644, 1460, 2013),
 (285536817.28, 188972.0829119788, 1511, 2014)]

## Healthcare Jobs

In [46]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%nurs%' OR JobTitle LIKE '%medical%' OR JobTitle LIKE '%dent%' OR JobTitle LIKE '%pharma%' 
OR JobTitle LIKE '%physician%' OR JobTitle LIKE '%social work%' Group by JobTitle Order by Year"""
cur.execute(query)
cur.fetchall()

[('ADMINISTRATOR, SFGH MEDICAL CENTER', 257124.44, 2011),
 ('ASSISTANT MEDICAL EXAMINER', 1063965.86, 2011),
 ('ASSISTANT SUPERINTENDENT RECREATION', 100515.4, 2011),
 ('BUILDINGS AND GROUNDS MAINTENANCE SUPERINTENDENT', 1617180.56, 2011),
 ('CAPTAIN, EMERGENCYCY MEDICAL SERVICES', 4176262.96, 2011),
 ('CHIEF NURSERY SPECIALIST', 84800.16, 2011),
 ('CITY SHOPS ASSISTANT SUPERINTENDENT', 211724.7, 2011),
 ('CLINICAL NURSE SPECIALIST', 3464581.28, 2011),
 ('CLINICAL PHARMACIST', 4326965.74, 2011),
 ('CONFIDENTIAL CHIEF ATTORNEY II (CIVIL & CRIMINAL)', 644966.04, 2011),
 ('CONFIDENTIAL SECRETARY CITY ATTORNEY', 78482.26, 2011),
 ('CONFIDENTIAL SECRETARY TO DISTRICT ATTORNEY', 81650.52, 2011),
 ('DENTAL AIDE', 839839.18, 2011),
 ('DENTAL HYGIENIST', 245250.97999999998, 2011),
 ('DENTIST', 940939.01, 2011),
 ('EMERGENCY MEDICAL SERVICES AGENCY SPECIALIST', 434689.06, 2011),
 ('INCIDENT SUPPORT SPECIALIST', 1085669.42, 2011),
 ('LICENSED VOCATIONAL NURSE', 11591302.27, 2011),
 ('MECHANICAL S